# Solver Analysis

In [ ]:
from inchworm import Solver
from triqs.plot.mpl_interface import *
from h5 import HDFArchive
%matplotlib inline

%pylab inline
pylab.rcParams['figure.figsize'] = (14, 8)

import pprint
pp = pprint.PrettyPrinter(indent=4)
pprint = pp.pprint

In [ ]:
with HDFArchive('Solver.h5','r') as arch:
    S = arch['S']
    
constr_params = S.constr_params
solve_params = S.last_solve_params

# Exact Propagator and Green Function Calculation (finite bath only)

In [ ]:
h_imp = solve_params['h_imp']

from model import *
    
if 'h_bath' in locals() and 'h_coup' in locals():
    
    h_tot = h_imp + h_bath + h_coup

    from triqs.atom_diag import *
    from inchworm.ad_tools import make_ED_propagator, create_effective_hyb
 
    gf_struct = constr_params['gf_struct']
    fop_imp = [(s,o) for s, n_orb in gf_struct for o in range(n_orb)]
    fop_bath = [(s,n_orb + o) for s, n_orb in gf_struct for o in range(n_orb_bath)]
    fop_tot = fop_imp + fop_bath
    
    eff_hyb = create_effective_hyb(gf_struct)
    
    ad_tot = AtomDiag(h_tot, fop_tot)
    
    gf_struct_tot = [[s, n_orb + n_orb_bath] for s, n_orb in gf_struct]
    G_tot = atomic_g_tau(ad_tot, beta, gf_struct_tot, constr_params['n_tau_green'])
    
    name_list = [bl for bl, n_orb in gf_struct]
    block_list = [G_tot[bl][:n_orb, :n_orb] for bl, n_orb in gf_struct]
    G_exact = BlockGf(name_list=name_list, block_list=block_list)
    
    if solve_params['n_bath_sites_ED'] == 0:
        ad_imp = AtomDiag(h_imp + S.h_bath_ED, eff_hyb, fop_imp)
        ad_bath = AtomDiag(h_bath, fop_bath)
        
        u_exact = make_ED_propagator(ad_tot, ad_imp, ad_bath, beta, constr_params['n_tau_inch'])

# Input Parameters

In [ ]:
pprint(constr_params)

# Construct Parameters

In [ ]:
pprint(solve_params)

# $\Delta(\tau)$

In [ ]:
oplot(S.Delta_tau, name='Delta')
if S.Delta_tau_ED is not None:
    oplot(S.Delta_tau_ED, name='Delta_ED')

In [ ]:
if S.Delta_tau_ED is not None:
    oplot(S.Delta_tau_tilde, name='')
    plt.title(r'$\tilde{\Delta}=\Delta - \Delta_{\rm ED}$',fontsize=18)

# ${\mathcal U}(\tau)$

In [ ]:
n_block = len(S.u_tau)

# Setup Subplots
from math import ceil
NX = 2
NY = ceil(n_block / NX)

plt.subplots(NY, NX, figsize=(16,6 * NY))
tau_max = constr_params['beta']

for n in range(n_block):
    plt.subplot(NY, NX, n + 1)
    
    bl = str(n)
    oplot(S.u_tau[bl], name='', x_window=(0,tau_max))
    if 'u_exact' in locals():
        oplot(u_exact[bl], 'x', color='grey', name='')
        
    plt.legend(loc='upper left')
    #plt.yscale('log')
    #plt.yscale('symlog')
    plt.title(r'u[bl={}]'.format(n))

# Deviations ${\mathcal U}(\tau)$

In [ ]:
relative = True

if 'u_exact' in locals():
    
    # Setup Subplots
    NX = 2
    NY = ceil(n_block / NX)
    plt.subplots(NY, NX, figsize=(16,6 * NY))
    tau_max = constr_params['beta']
    
    plt.suptitle(r'$\frac{u[bl](\tau) - u_{\rm exact}[bl](\tau)}{max(|u[bl](\tau)|,|u_{\rm exact}[bl](\tau)|)}$', fontsize=20)
    
    for n in range(n_block):
        plt.subplot(NY, NX, n + 1)
        
        bl = str(n)
        diff = S.u_tau[bl] - u_exact[bl]
        if relative: diff.data[:] = diff.data / np.maximum(np.abs(S.u_tau[bl].data), np.abs(u_exact[bl].data))
            
        oplot(diff, name='', x_window=(0,tau_max))
        
        plt.legend(loc='upper left')
        #plt.yscale('log')
        #plt.yscale('symlog')
        plt.title(r'bl={}'.format(n))

# ${\mathcal U}(\tau)$ by order

In [ ]:
if S.u_tau_by_order:
    
    n_order = len(S.u_tau_by_order)
    
    # Setup Subplots
    NX = 2
    NY = ceil(n_block * n_order/ NX)
    plt.subplots(NY, NX, figsize=(16,6 * NY))
    tau_max = constr_params['beta']

    for k, u_tau in enumerate(S.u_tau_by_order):
        for n in range(n_block):
            plt.subplot(NY, NX, n + k * n_block + 1)
            
            bl = str(n)
            oplot(u_tau[bl], x_window=(0,tau_max), marker='')
            
            plt.legend()
            #plt.yscale('log')
            #plt.yscale('symlog')
            plt.title(r'u[bl={}] order={}'.format(n, k))

# $G(\tau)$

In [ ]:
if S.G_tau is not None:
    
    gf_struct = constr_params['gf_struct']
    
    # Setup Subplots
    from math import ceil
    NX = 2
    NY = ceil(len(gf_struct) / NX)
    
    plt.subplots(NY, NX, figsize=(16,6 * NY))
    tau_max = constr_params['beta']
    
    for n, (bl, bl_size) in enumerate(gf_struct):
        plt.subplot(NY, NX, n + 1)
        
        oplot(S.G_tau[bl].real, name='', x_window=(0,tau_max))
        if 'G_exact' in locals():
            oplot(G_exact[bl].real, 'x', color='grey', name='')
            
        plt.legend()
        plt.title(r'G[bl={}]'.format(bl))

# Deviations $G(\tau)$

In [ ]:
relative = True

if 'G_exact' in locals():
    
    # Setup Subplots
    plt.subplots(NY, NX, figsize=(16,6 * NY))
    tau_max = constr_params['beta']
    
    plt.suptitle(r'$\frac{G[bl](\tau) - G_{\rm exact}[bl](\tau)}{max(|G[bl](\tau)|,|G_{\rm exact}[bl](\tau)|)}$', fontsize=20)
    
    for n, (bl, bl_size) in enumerate(gf_struct):
        plt.subplot(NY, NX, n + 1)
        
        diff = G_exact[bl] - S.G_tau[bl]
        if relative: diff.data[:] = diff.data / np.maximum(np.abs(S.G_tau[bl].data), np.abs(G_exact[bl].data))
            
        oplot(diff.real, name='', x_window=(0,tau_max))
        
        plt.legend()
        plt.title(r'bl={}'.format(n))

# Order Histogram

In [ ]:
for hist in S.order_histograms:
    plt.plot(hist)
plt.xlabel('order')
plt.ylabel('probability')

# Rerun

In [ ]:
# S2 = S.copy() # FIXME
S2 = Solver(**constr_params)
S2.Delta_tau << S.Delta_tau
#S2.solve(**solve_params)